# Explainable ML for DDoS detection (CSE-CIC-IDS2018): XGBoost + SHAP

COMP6310 group project. Single supervised model (XGBoost) that outputs a flag (benign / DDoS) and a confidence score.

**Data:** processed CICFlowMeter CSVs for 20 and 21 Feb 2018 (LOIC-HTTP on the 20th; HOIC and LOIC-UDP on the 21st).

**Things this notebook already accounts for (found while running it):**
- The Wednesday file has **no `Dst IP`**, so feature 7 (flows to the same destination in a 10 s window) cannot be built for HOIC / LOIC-UDP. It is switched off.
- LOIC-UDP has only 1,730 flows, so any LOIC-UDP result is noisy.
- Tool and day are tied together (LOIC-HTTP = Tuesday only; HOIC and LOIC-UDP = Wednesday only).

## 0. Imports

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from xgboost import XGBClassifier
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import train_test_split

pd.set_option("display.width", 200)

## 1. Settings (confirmed against the real files)

In [ ]:
DATA_DIR = Path("../data")      # put the CSV files here; do not commit them to GitHub
FILES = ["Tuesday-20-02-2018_TrafficForML_CICFlowMeter.csv",
         "Wednesday-21-02-2018_TrafficForML_CICFlowMeter.csv"]

LABEL, TIME, DST = "Label", "Timestamp", "Dst IP"
BASE_FEATURES = [
    "Flow Pkts/s", "Flow Byts/s", "Flow Duration",
    "Flow IAT Mean", "Flow IAT Std", "SYN Flag Cnt",
    "Fwd Pkt Len Mean", "Fwd Pkt Len Std",
]
COUNT_COLS = ["Tot Fwd Pkts", "Tot Bwd Pkts"]   # used to build the fwd/bwd ratio feature
USE_WINDOW_FEATURE = False   # feature 7 OFF: Wednesday file has no Dst IP
WINDOW = "10s"               # only used if USE_WINDOW_FEATURE is True
HELD_OUT_TOOL = "HOIC"       # one of: LOIC-HTTP, LOIC-UDP, HOIC

## 2. Load the data

The files are large, so only the needed columns are read. Some copies contain repeated header rows, so rows where the label equals "Label" are removed and numeric columns are forced to numbers.

In [ ]:
def load(path):
    head = pd.read_csv(path, nrows=0).columns.tolist()
    print(path.name, "-", len(head), "columns | has Dst IP:", DST in head)
    for need in [LABEL, TIME] + BASE_FEATURES + COUNT_COLS:
        if need not in head:
            print("   MISSING column:", need)
    wanted = [LABEL, TIME] + BASE_FEATURES + COUNT_COLS + ([DST] if USE_WINDOW_FEATURE else [])
    keep = [c for c in head if c in set(wanted)]
    df = pd.read_csv(path, usecols=keep, low_memory=False)
    df = df[df[LABEL] != LABEL].copy()
    for c in BASE_FEATURES + COUNT_COLS:
        if c in df:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    if TIME in df:
        df[TIME] = pd.to_datetime(df[TIME], dayfirst=True, errors="coerce")
    df["source_file"] = path.name
    return df

df = pd.concat([load(DATA_DIR / f) for f in FILES], ignore_index=True)
print(df.shape)

## 3. Inspect the labels

In [ ]:
print(df[LABEL].value_counts())
print(df.groupby("source_file")[LABEL].value_counts())
print("timestamp non-null share per file:")
print(df.groupby("source_file")[TIME].apply(lambda s: s.notna().mean()))

## 4. Keep benign and DDoS flows, and name the tool

In [ ]:
def tool_of(label):
    s = str(label).lower()
    if "benign" in s: return "Benign"
    if "udp" in s:    return "LOIC-UDP"
    if "hoic" in s:   return "HOIC"
    if "http" in s:   return "LOIC-HTTP"
    return "Other"

df["tool"] = df[LABEL].map(tool_of)
df = df[df["tool"] != "Other"].copy()
df["y"] = (df["tool"] != "Benign").astype(int)     # 1 = DDoS (positive class)
print(df["tool"].value_counts())

## 5. Clean (infinite, missing and duplicate values)

In [ ]:
n0 = len(df)
df = df.replace([np.inf, -np.inf], np.nan)
df = df.dropna(subset=BASE_FEATURES + COUNT_COLS)
df = df.drop_duplicates(subset=[c for c in df.columns if c != "source_file"])
print(f"rows: {n0} -> {len(df)}")
print(df["tool"].value_counts())

## 6. Build the features

`fwd_bwd_ratio` is derived from the packet counts. Identifiers (IP addresses, timestamps) are never model inputs. With the window feature off, the model has 9 input columns (8 base features plus the ratio).

In [ ]:
df["fwd_bwd_ratio"] = df["Tot Fwd Pkts"] / (df["Tot Bwd Pkts"] + 1)
FEATURES = BASE_FEATURES + ["fwd_bwd_ratio"]

if USE_WINDOW_FEATURE and DST in df and df[DST].notna().all() and TIME in df:
    df["_win"] = df[TIME].dt.floor(WINDOW)
    df["flows_to_dst"] = df.groupby([DST, "_win"])[DST].transform("size")
    FEATURES.append("flows_to_dst")
else:
    print("Window feature skipped (destination IP not available for every file).")
print("Model inputs:", FEATURES)

## 7. Held-out tool experiment

Train on the other tools, test on the held-out tool. Two ways to split the benign flows:

- `per_day`: within each day's file, the earlier 70% of benign flows go to training and the later 30% to testing.
- `by_file`: benign flows of the earlier day (Tuesday) train, the later day (Wednesday) test. Only meaningful when HOIC is held out, because HOIC is entirely on Wednesday.

Training benign flows are capped at 1,000,000 (random sample) for speed. The test set uses all held-out benign flows. Class imbalance is handled with `scale_pos_weight` (no SMOTE yet).

In [ ]:
def run(held_out, benign_mode="per_day", train_frac=0.7, benign_cap=1_000_000, seed=0):
    d = df[df[TIME].notna()]
    ben = d[d["tool"] == "Benign"]
    att = d[d["tool"] != "Benign"]

    if benign_mode == "per_day":
        pct = ben.groupby("source_file")[TIME].rank(pct=True, method="first")
        ben_tr, ben_te = ben[pct <= train_frac], ben[pct > train_frac]
    else:  # "by_file"
        order = ben.groupby("source_file")[TIME].min().sort_values().index
        ben_tr = ben[ben["source_file"] == order[0]]
        ben_te = ben[ben["source_file"] == order[1]]

    tr = pd.concat([ben_tr.sample(min(len(ben_tr), benign_cap), random_state=seed),
                    att[att["tool"] != held_out]])
    te = pd.concat([ben_te, att[att["tool"] == held_out]])

    spw = (tr["y"] == 0).sum() / (tr["y"] == 1).sum()
    model = XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1,
                          tree_method="hist", scale_pos_weight=spw,
                          n_jobs=-1, random_state=seed)
    model.fit(tr[FEATURES], tr["y"])

    p = model.predict_proba(te[FEATURES])[:, 1]
    pred, y = p >= 0.5, te["y"].values
    print(f"held out {held_out} | benign split: {benign_mode}")
    print(f"  train: {(tr.y==0).sum():,} benign, {(tr.y==1).sum():,} attack")
    print(f"  test:  {(y==0).sum():,} benign, {(y==1).sum():,} attack")
    print(f"  recall on {held_out}: {pred[y==1].mean():.3f}")
    print(f"  false positive rate on benign: {pred[y==0].mean():.4f}  ({int(pred[y==0].sum()):,} false alarms)")
    print(f"  PR-AUC: {average_precision_score(y, p):.3f}  (base rate {y.mean():.3f})")
    return model, te

m1, te1 = run("HOIC", "per_day")
m2, te2 = run("HOIC", "by_file")
m3, te3 = run("LOIC-HTTP", "per_day")

## 8. Diagnostics: score distribution, thresholds, feature importance

Read the AUC numbers with care. When most attack scores sit near 0, a very high ROC-AUC or PR-AUC can come from tiny differences in the ranking, so report **recall at the fixed 0.5 threshold** as the main number.

In [ ]:
def diagnose(name, model, te):
    p = model.predict_proba(te[FEATURES])[:, 1]
    y = te["y"].values
    print(name, "| ROC-AUC:", round(roc_auc_score(y, p), 4))
    print("  attack score 10/50/90th pct:", np.percentile(p[y == 1], [10, 50, 90]).round(3))
    print("  benign score 50/99/99.9th pct:", np.percentile(p[y == 0], [50, 99, 99.9]).round(4))
    for t in [0.5, 0.2, 0.1, 0.05, 0.01]:
        pr = p >= t
        print(f"  threshold {t}: recall {pr[y==1].mean():.3f}, FPR {pr[y==0].mean():.4f}")

diagnose("HOIC held out, per_day", m1, te1)
diagnose("HOIC held out, by_file", m2, te2)
diagnose("LOIC-HTTP held out, per_day", m3, te3)

print(te2.groupby("tool")[FEATURES].median().T)
print({k: round(v, 1) for k, v in m2.get_booster().get_score(importance_type="gain").items()})

## 9. Profile of each attack tool

In [ ]:
att = df[df["tool"] != "Benign"]
print((att["Fwd Pkt Len Mean"] == 0).groupby(att["tool"]).mean().rename("share with zero fwd payload"))
print(att.groupby("tool")[FEATURES].median().T.round(1))
print((att["SYN Flag Cnt"] != 0).groupby(att["tool"]).mean().rename("share with non-zero SYN Flag Cnt"))

## 10. In-distribution baseline

Random split with all tools seen in training. This is an **upper bound**: flows from the same attack burst appear in both train and test, so near-duplicates inflate it. Comparing it with section 7 separates "the features cannot detect this attack" from "the model cannot transfer to an unseen tool".

In [ ]:
def baseline(seed=0, benign_n=1_000_000):
    ben = df[df["tool"] == "Benign"].sample(benign_n, random_state=seed)
    d = pd.concat([ben, df[df["tool"] != "Benign"]])
    tr, te = train_test_split(d, test_size=0.3, stratify=d["tool"], random_state=seed)
    spw = (tr["y"] == 0).sum() / (tr["y"] == 1).sum()
    model = XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1, tree_method="hist",
                          scale_pos_weight=spw, n_jobs=-1, random_state=seed).fit(tr[FEATURES], tr["y"])
    return model, te

mb, teb = baseline(seed=0)
pred = mb.predict_proba(teb[FEATURES])[:, 1] >= 0.5
yb = teb["y"].values
print("baseline FPR:", round(pred[yb == 0].mean(), 4), f"({int(pred[yb == 0].sum()):,} false alarms)")
for t in ["HOIC", "LOIC-HTTP", "LOIC-UDP"]:
    m = (teb["tool"] == t).values
    print(f"baseline recall {t}: {pred[m].mean():.3f} (n={m.sum():,})")

## 11. SHAP explanations (not yet run: check it works before relying on it)

Run `pip install shap` once if the import fails. Part A shows which features drive the baseline model's decisions for each tool (mean absolute SHAP value). Part B retrains the baseline with several seeds and checks whether the feature ranking is stable (the Week 4 stability check). Benign flows are capped at 300,000 per seed here to keep it quick.

In [ ]:
import shap

def mean_abs_shap(model, te, per_tool=2000, seed=0):
    explainer = shap.TreeExplainer(model)
    out = {}
    for t in ["Benign", "HOIC", "LOIC-HTTP", "LOIC-UDP"]:
        s = te[te["tool"] == t]
        s = s.sample(min(per_tool, len(s)), random_state=seed)
        sv = explainer.shap_values(s[FEATURES])
        out[t] = np.abs(sv).mean(axis=0)
    return pd.DataFrame(out, index=FEATURES)

# Part A: which features matter for which tool (baseline model from section 10)
imp = mean_abs_shap(mb, teb)
print(imp.round(3))

In [ ]:
# Part B: stability across random seeds (attack tools averaged)
SEEDS = [0, 1, 2, 3, 4]
per_seed = {}
for s in SEEDS:
    m_s, te_s = baseline(seed=s, benign_n=300_000)
    imp_s = mean_abs_shap(m_s, te_s, seed=s)
    per_seed[s] = imp_s[["HOIC", "LOIC-HTTP", "LOIC-UDP"]].mean(axis=1)
    print("seed", s, "top 3:", list(per_seed[s].sort_values(ascending=False).index[:3]))

stab = pd.DataFrame(per_seed)
print(stab.round(3))
print("Spearman rank correlation between seeds:")
print(stab.corr(method="spearman").round(2))

## 12. Findings so far (for the Week 4 write-up)

- **Cleaning:** 8,997,323 -> 8,872,017 rows (1.4% removed). LOIC-UDP unchanged at 1,730.
- **Baseline, all tools seen in training:** recall HOIC 0.999, LOIC-HTTP 0.987, LOIC-UDP 1.000 (519 test flows), FPR 0.0084. Treat as an upper bound.
- **Held-out HOIC (train LOIC-HTTP + LOIC-UDP):** recall 0.156 (per_day) / 0.195 (by_file) at threshold 0.5; at most about 0.40 at any threshold tried.
- **Held-out LOIC-HTTP:** recall 0.003, FPR 0.0054.
- **Interpretation:** the features can detect each tool when the model has seen it, but the model does not transfer to an unseen tool. The three tools differ by orders of magnitude in duration, packet rate and payload, so two tools give no shared signature to learn.
- **Limitations:** feature 7 dropped (no `Dst IP` on Wednesday); LOIC-UDP is tiny; tool and day are confounded; random-split baseline is optimistic; `SYN Flag Cnt` had median 0 for every tool and no gain in the held-out model.